# scRNA-seq

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
import scvi
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

scvi.settings.seed = 30

Seed set to 30


In [2]:
#root_dir = "" ### <--- root directory path
root_dir = "/home/wcjohnchen2022/shared/github_single_cell_rnaseq/data/breast_cancer" ### <--- root directory path
dir = Path(root_dir)
ribo_url = "http://software.broadinstitute.org/gsea/msigdb/download_geneset.jsp?geneSetName=KEGG_RIBOSOME&fileType=txt"

In [3]:
def doublet_detection(data, file):
    scvi.model.SCVI.setup_anndata(data)
    vae = scvi.model.SCVI(data) ### building a single-cell variational inference model for single-cell omics analysis
    vae.train() ### training model.  (optional) modify parameter max_epochs
    solo = scvi.external.SOLO.from_scvi_model(vae) ### buiding a SOLO model from the trained SCVI model, for doublet detection
                                            ### assessing learning outcomes: extract learned representation and categorize by a classifier
    solo.train() ### training model.  (optional) modify parameter max_epochs
    df = solo.predict() ### predict doublets
    
    df['prediction'] = solo.predict(soft = False) ### make a new column to retain the labeled information
    df.index = df.index.map(lambda x: x[:-2]) ### <--- may need to modify depending on index label
    df['dif'] = df.doublet - df.singlet
    doublets = df[(df.prediction == 'doublet') & (df.dif > ((max(df.dif))*0.90))] ### <--- setting threshold

    data = sc.read_csv(file).T
    data.obs['Sample'] = file.split('/')[7] ### <--- may need to modify the number for splitting "/" depending on directory path
    data.obs['doublet'] = data.obs.index.isin(doublets.index)
    data = data[~data.obs.doublet]
    return data

def processing(file):
    data = sc.read_csv(file).T ### scanpy: row = cells, column = genes

    ############################### remove cells that do not meet the required minium number of genes
    sc.pp.filter_cells(data, min_genes = 200)

    ############################### remove genes that do not meet the required minium number of cells
    sc.pp.filter_genes(data, min_cells = 100)

    ############################### perform doublet detection using predictive generative model with default parameters
    data = doublet_detection(data, file) 

    ############################### filter for gene counts, remove mitochondrial genes and ribosomal genes
    data.var['mt'] = data.var.index.str.startswith('MT-')
    ribo_genes = pd.read_table(ribo_url, skiprows=2, header = None)
    data.var['ribo'] = data.var_names.isin(ribo_genes[0].values)
    sc.pp.calculate_qc_metrics(data, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True)
    upper_lim = np.quantile(data.obs.n_genes_by_counts.values, .98)
    lower_lim = np.quantile(data.obs.n_genes_by_counts.values, .02)
    data = data[(data.obs.n_genes_by_counts < upper_lim) & (data.obs.n_genes_by_counts > lower_lim)]
    data = data[data.obs.pct_counts_mt < 20]
    data = data[data.obs.pct_counts_ribo < 20]
    return data

In [4]:
############################### process data
output=[]
for folder in dir.rglob("*"):
    if folder.is_dir():
        mat = os.path.join(root_dir, folder, 'matrix.csv')
        output.append(processing(mat))

############################### concatenate data
dat = sc.concat(output)

############################### save data as sparse matrix
dat.X = csr_matrix(dat.X)
dat.write_h5ad('scrnaseq_processed_data_10232025.h5ad')

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Training:   0%|          | 0/400 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=400` reached.


INFO     Creating doublets, preparing SOLO model.                                                                  


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


...

`Trainer.fit` stopped: `max_epochs=400` reached.


INFO     Creating doublets, preparing SOLO model.                                                                  


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Monitored metric validation_loss did not improve in the last 30 records. Best score: 0.293. Signaling Trainer to stop.


In [10]:
print(dat.obs)
print(dat.X)
print(dat.obs['Sample'].unique())

                                Sample  doublet  n_genes_by_counts  \
CID3941_ACAGCCGAGCTCTCGG    ER_CID3941    False               2456   
CID3941_ACGCAGCCAGCTGTAT    ER_CID3941    False               2305   
CID3941_CATGACAGTAGAGCTG    ER_CID3941    False               1944   
CID3941_CGAACATTCCCAACGG    ER_CID3941    False               1492   
CID3941_CTAGAGTGTAAAGGAG    ER_CID3941    False               1353   
...                                ...      ...                ...   
CID4465_CGTGTCTAGTAAGTAC  TNBC_CID4465    False               1461   
CID4465_GAGTCCGGTGAGGGAG  TNBC_CID4465    False               1090   
CID4465_GTAACGTCAAGCCATT  TNBC_CID4465    False                268   
CID4465_TGAGCCGCACGTCTCT  TNBC_CID4465    False               1324   
CID4465_TGGCCAGGTCCGTGAC  TNBC_CID4465    False                552   

                          total_counts  total_counts_mt  pct_counts_mt  \
CID3941_ACAGCCGAGCTCTCGG        6685.0            578.0       8.646223   
CID3941_ACG